In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st

In [2]:
# Read data

data=pd.read_csv('Churn_Modelling.csv')
data.head(6)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1


In [3]:
# Remove UnNecessary columns
data.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [4]:
data.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# Apply Encoding for Categorical Variables
#Import libraries
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder

In [6]:
# We will aply Label Encoder for Gender because it has only 2 unique Values and OneHot Encoder for Geography Column , Before applying perform basic Feature ENgineering
data.describe() # Gives output only for Numerical columns
data.info()  # Provides the datatype about each column
data.shape # 10000 rows with 11 columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


(10000, 11)

In [7]:
# LabelEncoder on Categorical Variables
data['Gender'].unique() # 2 unique values Male & Female
label_encoder_gender=LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [8]:
data.head(10)  # Gender has been encoded with 0,1

print(" The gender values after encoding :",data['Gender'].unique())
data['Gender'].value_counts()

 The gender values after encoding : [0 1]


Gender
1    5457
0    4543
Name: count, dtype: int64

In [9]:
# Apply Onehot encoding for Geography Column

data['Geography'].unique()
onehot_encoder_geo=OneHotEncoder()
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

In [10]:
onehot_encoder_geo.get_feature_names_out()
geo_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out())

In [11]:
# Concatenate with actual table 
data=pd.concat([data,geo_df],axis=1).drop('Geography',axis=1) ## Dropped the Geograpy column and replaced it with 3 one hot encoded columns

In [12]:
# Create pickle file
import pickle

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

In [13]:
data.head(10)  # Final Dataframe ready for model prediction

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0
8,501,1,44,4,142051.07,2,0,1,74940.50,0,1.0,0.0,0.0
9,684,1,27,2,134603.88,1,1,1,71725.73,0,1.0,0.0,0.0


In [14]:
# Split into train and test data and appply standard scaler for training data 

X=data.drop('Exited',axis=1)
y=data['Exited']

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [15]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((7000, 12), (3000, 12), (7000,), (3000,))

In [16]:
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [17]:
# Create a pickle file for scaler 
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [18]:
# Create a ANN model 
# First import libraries
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [19]:
model= Sequential(
    [
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(64,activation='relu'),
    Dense(1,activation='sigmoid')
    ]
)

In [20]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 64)                4160      
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 5057 (19.75 KB)
Trainable params: 5057 (19.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [22]:
callbacks_earlystopping=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [23]:
import datetime
log_dir='logs/fit/'+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
callbacks_tensorboard=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [24]:
# Train the model 

model_training=model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=50,callbacks=[callbacks_tensorboard,callbacks_earlystopping])

Epoch 1/50


219/219 [==============================] - 2s 3ms/step - loss: 0.4029 - accuracy: 0.8327 - val_loss: 0.3490 - val_accuracy: 0.8580
Epoch 2/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3568 - accuracy: 0.8543 - val_loss: 0.3320 - val_accuracy: 0.8690
Epoch 3/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3529 - accuracy: 0.8537 - val_loss: 0.3491 - val_accuracy: 0.8550
Epoch 4/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3519 - accuracy: 0.8577 - val_loss: 0.3523 - val_accuracy: 0.8600
Epoch 5/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3441 - accuracy: 0.8593 - val_loss: 0.3388 - val_accuracy: 0.8663
Epoch 6/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3388 - accuracy: 0.8600 - val_loss: 0.3371 - val_accuracy: 0.8667
Epoch 7/50
219/219 [==============================] - 0s 2ms/step - loss: 0.3388 - accuracy: 0.8590 - val_loss: 0.3361 - val_accuracy: 0.8683


In [25]:
model.save('model.h5')

c:\Sowjanya\Datasets\CustomerChurnModelPrediction\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# Setup tensorboard
%load_ext tensorboard

In [ ]:
##%tensorboard --logdir logs/fit##

In [ ]:
# Due to above error , I ran the below command in terminal 
###python -m tensorboard.main --logdir=logs/fit